# Medical Data With ifcfill And SeqTree

This notebook demonstrates a fuller workflow:

1. Load public Synthea healthcare sample data from a URL.
2. Keep a patient-level table with dates, categorical values, numeric values, and missing values.
3. Use `ifcfill` to impute missing values and convert datetimes into numeric IFC columns.
4. Encode pandas categorical columns as integer category codes for SeqTree.
5. Fit SeqTree, sample the same number of rows, decode categories, and pass the result through `ifcfill.inverse_transform()`.

The Synthea sample data is generated healthcare data, not real patient PHI. That makes it suitable for a public tutorial while still looking like a medical table.

In [ ]:
# Run one of these once in a fresh notebook environment.
#
# From PyPI:
# %pip install -q "seqtree[examples]"
#
# From a local SeqTree checkout:
# %pip install -q -e ".[examples]"

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import sys

import pandas as pd

# When running this notebook from the repository checkout, prefer the local SeqTree source tree.
repo_root = Path.cwd()
if (repo_root / "src" / "seqtree").exists():
    sys.path.insert(0, str(repo_root / "src"))

from ifcfill import IFCTransformer
from seqtree import SequentialTreeSynthesizer

In [ ]:
DATA_URL = "https://github.com/synthetichealth/synthea-sample-data/raw/main/downloads/synthea_sample_data_csv_apr2020.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "synthea_sample_data_csv_apr2020.zip"

DATA_DIR.mkdir(exist_ok=True)
if not ZIP_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

with ZipFile(ZIP_PATH) as zf:
    csv_names = zf.namelist()
    patients_name = next(name for name in csv_names if name.endswith("patients.csv"))
    with zf.open(patients_name) as f:
        raw_patients = pd.read_csv(f)

raw_patients.shape, raw_patients.head()

In [ ]:
# A compact but mixed-type patient table: dates, strings, floats, and missing DEATHDATE values.
columns = [
    "BIRTHDATE",
    "DEATHDATE",
    "MARITAL",
    "RACE",
    "ETHNICITY",
    "GENDER",
    "BIRTHPLACE",
    "CITY",
    "STATE",
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE",
]

patients = raw_patients[columns].copy()
patients["BIRTHDATE"] = pd.to_datetime(patients["BIRTHDATE"], errors="coerce")
patients["DEATHDATE"] = pd.to_datetime(patients["DEATHDATE"], errors="coerce")

# Keep the example quick for notebooks while preserving enough rows for synthesis.
patients = patients.sample(min(len(patients), 1000), random_state=42).reset_index(drop=True)

patients.head(), patients.isna().sum().sort_values(ascending=False).head(8)

In [ ]:
ifc = IFCTransformer(
    col_types={
        "BIRTHDATE": "datetime",
        "DEATHDATE": "datetime",
        "MARITAL": "categorical",
        "RACE": "categorical",
        "ETHNICITY": "categorical",
        "GENDER": "categorical",
        "BIRTHPLACE": "categorical",
        "CITY": "categorical",
        "STATE": "categorical",
        "HEALTHCARE_EXPENSES": "float",
        "HEALTHCARE_COVERAGE": "float",
    },
    int_fill="median",
    float_fill="median",
    cat_fill="mode",
    datetime_anchor="1900-01-01",
    datetime_unit="D",
)

ifc_table = ifc.fit_transform(patients)

ifc.missing_report_.sort_values("missing_fraction", ascending=False).head(10), ifc_table.head()

`ifcfill` has done the missing-value work and converted dates to numeric offsets. For SeqTree, categorical columns need integer category codes, so the next cell records each category vocabulary and replaces the pandas categorical columns with their codes.

In [ ]:
seqtree_input = ifc_table.copy()
category_levels = {}

for column, col_type in ifc.column_types_.items():
    if col_type == "categorical":
        categorical = seqtree_input[column].astype("category")
        category_levels[column] = list(categorical.cat.categories)
        seqtree_input[column] = categorical.cat.codes.astype("int64")

continuous_columns = [
    column
    for column, col_type in ifc.column_types_.items()
    if col_type in {"float", "datetime"} and column in seqtree_input.columns
]
discrete_columns = [
    column
    for column, col_type in ifc.column_types_.items()
    if col_type in {"integer", "categorical"} and column in seqtree_input.columns
]

# Datetime columns came from ifcfill as integer day offsets. Cast them to float because
# we want SeqTree to interpolate them as continuous timeline variables.
seqtree_input[continuous_columns] = seqtree_input[continuous_columns].astype("float64")
seqtree_input[discrete_columns] = seqtree_input[discrete_columns].astype("int64")

continuous_columns, discrete_columns, seqtree_input.head()

In [ ]:
model = SequentialTreeSynthesizer(
    optimize_order=True,
    tree_backend="auto",
    continuous_strategy="interpolate",
    continuous_columns=continuous_columns,
    discrete_columns=discrete_columns,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)

model.fit(seqtree_input)
synthetic_ifc_codes = model.sample(len(seqtree_input), random_state=101, as_dataframe=True, n_jobs=-1)

model.get_variable_order(), synthetic_ifc_codes.head()

In [ ]:
synthetic_ifc = synthetic_ifc_codes.copy()

for column, levels in category_levels.items():
    codes = synthetic_ifc[column].round().astype("int64").clip(0, len(levels) - 1)
    synthetic_ifc[column] = pd.Categorical.from_codes(codes, categories=levels)

# Round datetime offsets to whole days before passing them back through ifcfill.
for column, col_type in ifc.column_types_.items():
    if col_type == "datetime" and column in synthetic_ifc.columns:
        synthetic_ifc[column] = synthetic_ifc[column].round().astype("int64")

synthetic_patients = ifc.inverse_transform(
    synthetic_ifc,
    restore_missing=True,
    random_state=202,
)

synthetic_patients.head(), synthetic_patients.shape

In [ ]:
summary = pd.DataFrame(
    {
        "original_missing": patients.isna().mean(),
        "synthetic_missing": synthetic_patients.isna().mean(),
    }
)

summary.sort_values("original_missing", ascending=False)

This example keeps the same row count as the input table. For real projects, inspect utility and privacy risk before publishing synthetic data, especially when the source resembles patient-level records.